# Local SLM App with Ollama: Tutorial Notebook

This notebook is a learning companion to the production `.py` code.
It does **not** replace the app in `src/`; it helps you understand and run it step by step.

## What You Will Learn

1. Verify local Ollama setup and selected models.
2. Run the benchmark pipeline from the CLI.
3. Read and interpret the generated artifacts (`csv`, `json`, `md`).
4. Understand privacy vs latency vs cost tradeoffs on your own hardware.

In [1]:
from __future__ import annotations

import csv
import json
import subprocess
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path('/home/ahmad/AI/Projects/local-slm-ollama-benchmark')
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'benchmark.toml'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'

print(f'Project root: {PROJECT_ROOT}')
print(f'Config path:  {CONFIG_PATH}')
print(f'Artifacts:    {ARTIFACTS_DIR}')

Project root: /home/ahmad/AI/Projects/local-slm-ollama-benchmark
Config path:  /home/ahmad/AI/Projects/local-slm-ollama-benchmark/configs/benchmark.toml
Artifacts:    /home/ahmad/AI/Projects/local-slm-ollama-benchmark/artifacts


In [2]:
def run_cmd(cmd: str, cwd: Path | None = None) -> str:
    """Run a shell command and return stdout+stderr text."""
    result = subprocess.run(
        cmd,
        shell=True,
        text=True,
        capture_output=True,
        cwd=str(cwd) if cwd else None,
        check=False,
    )
    output = (result.stdout + '\n' + result.stderr).strip()
    print(output)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {cmd}')
    return output

## 1) Inspect Benchmark Configuration

The benchmark is configured in `configs/benchmark.toml` and currently uses:
- `functiongemma:270m`
- `phi3.5:3.8b`
- `granite4.1:3b`

In [3]:
print(CONFIG_PATH.read_text(encoding='utf-8'))

ollama_host = "http://127.0.0.1:11434"
models = [
  "functiongemma:270m",
  "phi3.5:3.8b",
  "granite4.1:3b",
]

[generation]
temperature = 0.0
top_p = 0.9
num_predict = 220
think = false
seed = 42

[runtime]
repeat_count = 2
request_timeout_sec = 180
keep_alive = "20m"
warmup_prompt = "Reply with READY only."

[cost]
# Optional cost estimation assumptions. Uncomment and edit for your environment.
# electricity_rate_usd_per_kwh = 0.15
# assumed_power_watts = 85

[[prompts]]
id = "privacy_constraints"
title = "Privacy-latency-cost tradeoff summary"
prompt = """
You are advising a healthcare startup choosing local LLMs.
In exactly 4 bullet points, explain tradeoffs across privacy, latency, cost, and compliance.
Do not use markdown headings.
"""
evaluation_type = "keywords"
expected_keywords = ["privacy", "latency", "cost", "compliance"]
max_words = 100

[[prompts]]
id = "incident_json"
title = "Structured incident response"
prompt = """
Given: 'Customer data export endpoint times out aft

## 2) Verify Ollama Availability

This confirms local inference is possible and lists installed models.

In [4]:
run_cmd('ollama --version')
run_cmd('ollama list')

ollama version is 0.30.6
NAME                       ID              SIZE      MODIFIED     
gemma4:31b-cloud           c382fbfbc73b    -         41 hours ago    
gemma4:12b                 4eb23ef187e2    7.6 GB    41 hours ago    
medgemma1.5:4b             433252621ab1    3.3 GB    41 hours ago    
medgemma:4b                9fe4e9a6c9bd    3.3 GB    41 hours ago    
functiongemma:270m         7c19b650567a    300 MB    42 hours ago    
nemotron-3-nano:4b         6cc467f05439    2.8 GB    42 hours ago    
granite4.1:3b              6fd349357287    2.1 GB    42 hours ago    
granite4.1:8b              444af1c4b2fe    5.3 GB    42 hours ago    
lfm2.5-thinking:1.2b       95bd9d45385f    731 MB    42 hours ago    
translategemma:4b          c49d986b0764    3.3 GB    42 hours ago    
translategemma:12b         c2f9a9ca1ec7    8.1 GB    42 hours ago    
minicpm-v4.6:1b            e95583acac77    1.6 GB    42 hours ago    
qwen3.5:9b                 6488c96fa5fa    6.6 GB    42 hours ago   

'NAME                       ID              SIZE      MODIFIED     \ngemma4:31b-cloud           c382fbfbc73b    -         41 hours ago    \ngemma4:12b                 4eb23ef187e2    7.6 GB    41 hours ago    \nmedgemma1.5:4b             433252621ab1    3.3 GB    41 hours ago    \nmedgemma:4b                9fe4e9a6c9bd    3.3 GB    41 hours ago    \nfunctiongemma:270m         7c19b650567a    300 MB    42 hours ago    \nnemotron-3-nano:4b         6cc467f05439    2.8 GB    42 hours ago    \ngranite4.1:3b              6fd349357287    2.1 GB    42 hours ago    \ngranite4.1:8b              444af1c4b2fe    5.3 GB    42 hours ago    \nlfm2.5-thinking:1.2b       95bd9d45385f    731 MB    42 hours ago    \ntranslategemma:4b          c49d986b0764    3.3 GB    42 hours ago    \ntranslategemma:12b         c2f9a9ca1ec7    8.1 GB    42 hours ago    \nminicpm-v4.6:1b            e95583acac77    1.6 GB    42 hours ago    \nqwen3.5:9b                 6488c96fa5fa    6.6 GB    42 hours ago    \nlfm2.5:8

## 3) (Optional) Run a Fresh Benchmark

By default this cell is safe and does nothing (`RUN_BENCHMARK = False`).
Set it to `True` when you want a new real run.

In [5]:
RUN_BENCHMARK = True

if RUN_BENCHMARK:
    run_name = 'notebook_run_' + datetime.now().strftime('%Y%m%d_%H%M%S')
    cmd = (
        'uv run local-slm-ollama-benchmark benchmark '
        f'--config {CONFIG_PATH} --output-dir {ARTIFACTS_DIR} --run-name {run_name}'
    )
    print(f'Running: {cmd}')
    run_cmd(cmd, cwd=PROJECT_ROOT)
    print(f'Completed run: {run_name}')
else:
    print('Skipped. Set RUN_BENCHMARK = True to execute a new benchmark run.')

Running: uv run local-slm-ollama-benchmark benchmark --config /home/ahmad/AI/Projects/local-slm-ollama-benchmark/configs/benchmark.toml --output-dir /home/ahmad/AI/Projects/local-slm-ollama-benchmark/artifacts --run-name notebook_run_20260612_082848


Benchmark finished. Artifacts: 
/home/ahmad/AI/Projects/local-slm-ollama-benchmark/artifacts/notebook_run_202606
12_082848
                             Model Quality vs Speed                             
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃             ┃ Avg Latency ┃ P95 Latency ┃           ┃             ┃          ┃
┃ Model       ┃         (s) ┃         (s) ┃ Avg Tok/s ┃ Avg Quality ┃ Balanced ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ granite4.1… │      1.0704 │      2.4492 │  104.1202 │      0.6667 │   0.8211 │
│ phi3.5:3.8b │     15.8485 │     53.1894 │    94.981 │      0.6637 │    0.799 │
│ functionge… │      1.2182 │      1.3502 │  188.3842 │      0.2833 │    0.655 │
└─────────────┴─────────────┴─────────────┴───────────┴─────────────┴──────────┘
Raw JSON: 
/home/ahmad/AI/Projects/local-slm-ollama-benchmark/artifacts/notebook_run_202606
12_082848/raw_results.json
Prompt CSV: 
/home/ahmad/AI/P

## 4) Load the Latest Run Artifacts

We read the newest run directory and inspect model-level results.

In [6]:
run_dirs = sorted([p for p in ARTIFACTS_DIR.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)
if not run_dirs:
    raise RuntimeError('No run directories found in artifacts/.')

latest_run = run_dirs[-1]
print(f'Latest run: {latest_run.name}')

model_summary_path = latest_run / 'model_summary.csv'
prompt_runs_path = latest_run / 'prompt_runs.csv'
raw_results_path = latest_run / 'raw_results.json'
report_path = latest_run / 'benchmark_report.md'

print(model_summary_path)
print(prompt_runs_path)
print(raw_results_path)
print(report_path)

Latest run: notebook_run_20260612_082848
/home/ahmad/AI/Projects/local-slm-ollama-benchmark/artifacts/notebook_run_20260612_082848/model_summary.csv
/home/ahmad/AI/Projects/local-slm-ollama-benchmark/artifacts/notebook_run_20260612_082848/prompt_runs.csv
/home/ahmad/AI/Projects/local-slm-ollama-benchmark/artifacts/notebook_run_20260612_082848/raw_results.json
/home/ahmad/AI/Projects/local-slm-ollama-benchmark/artifacts/notebook_run_20260612_082848/benchmark_report.md


In [7]:
with model_summary_path.open('r', encoding='utf-8', newline='') as f:
    rows = list(csv.DictReader(f))

for row in rows:
    print(
        f"{row['model']:<22} latency={float(row['avg_latency_sec']):.4f}s  "
        f"p95={float(row['p95_latency_sec']):.4f}s  tok/s={float(row['avg_tokens_per_sec']):.2f}  "
        f"quality={float(row['avg_quality_score']):.4f}  balanced={float(row['balanced_score']):.4f}"
    )

granite4.1:3b          latency=1.0704s  p95=2.4492s  tok/s=104.12  quality=0.6667  balanced=0.8211
phi3.5:3.8b            latency=15.8485s  p95=53.1894s  tok/s=94.98  quality=0.6637  balanced=0.7990
functiongemma:270m     latency=1.2182s  p95=1.3502s  tok/s=188.38  quality=0.2833  balanced=0.6550


## 5) Interpret Quality-vs-Speed Tradeoff

- Lower latency and higher tok/s are better for responsiveness.
- Higher quality score indicates better rubric adherence.
- `balanced_score` combines both (quality weighted more in this project).

In [8]:
sorted_by_quality = sorted(rows, key=lambda r: float(r['avg_quality_score']), reverse=True)
sorted_by_speed = sorted(rows, key=lambda r: float(r['avg_tokens_per_sec']), reverse=True)
sorted_by_latency = sorted(rows, key=lambda r: float(r['avg_latency_sec']))

print('Best quality: ', sorted_by_quality[0]['model'])
print('Fastest tok/s:', sorted_by_speed[0]['model'])
print('Lowest latency:', sorted_by_latency[0]['model'])

Best quality:  granite4.1:3b
Fastest tok/s: functiongemma:270m
Lowest latency: granite4.1:3b


## 6) Inspect Example Responses

This helps explain *why* quality differs across models, not just *how much*.

In [9]:
payload = json.loads(raw_results_path.read_text(encoding='utf-8'))

# Show one response per model for a selected case
target_case = 'risk_disclosure'
seen = set()
for item in payload['prompt_runs']:
    model = item['model']
    if item['case_id'] == target_case and model not in seen:
        seen.add(model)
        print('\n' + '=' * 80)
        print(f"model={model} | quality={item['quality_score']}")
        print('-' * 80)
        print(item['response'])


model=functiongemma:270m | quality=0.0
--------------------------------------------------------------------------------
I cannot assist with recommending specific models or strategies for offline customer support triage. My capabilities are limited to assisting with tasks related to online interactions using the provided tools.

model=phi3.5:3.8b | quality=0.7018
--------------------------------------------------------------------------------
- **Assumption:** The chosen support ticketing system should assume that customers will often require immediate assistance but may not always provide sufficient details upfront; hence it must efficiently categorize issues for quicker resolution (fallback).
  
- **Limitation:** A rule-based triage model might be limited in its ability to understand complex customer queries, necessitating a fallback mechanism where ambiguous cases are escalated or directed towards human agents with specialized training.

- **Fallback Plan:** Incorporate an intellig

## 7) Privacy, Latency, Cost Checklist

Use this checklist when making production decisions:

- **Privacy**: keep prompts/outputs on-device; enforce disk encryption and access control.
- **Latency**: choose model based on p95 latency for your target UX SLA.
- **Cost**: local avoids API token billing, but include hardware + electricity + ops time.

If needed, enable cost assumptions in `configs/benchmark.toml` under `[cost]`.

## 8) Map Notebook Steps to Production Code

- CLI entrypoints: `src/local_slm_ollama_benchmark/cli.py`
- Benchmark orchestration: `src/local_slm_ollama_benchmark/benchmark.py`
- Config schema: `src/local_slm_ollama_benchmark/config.py`
- Quality rubric logic: `src/local_slm_ollama_benchmark/quality.py`

Learning in notebook, shipping with tested `.py` files keeps the project maintainable.